In [1]:
import joblib

import math
import re

from collections import Counter

import nltk
import pandas as pd

from nltk.util import ngrams
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords

In [2]:
# Calculate entropy
def calculate_entropy(freq_dict, total_count):
    return -sum((count / total_count) * math.log2(count / total_count) for count in freq_dict.values() if count > 0)

def calculate_features(text):
    # Tokenize the text
    try:
        words = word_tokenize(text)
    except TypeError as exc:
        print(text)
        raise exc
    
    num_words = len(words)
    num_chars = len(text)
    num_lines = text.count('\n') + 1

    # Calculate word frequency
    word_freq = Counter(words)

    # Calculate n-grams (bigrams in this example)
    bigrams = list(ngrams(words, 2))
    bigram_freq = Counter(bigrams)

    # Calculate character distribution
    char_freq = Counter(text)

    # Calculate punctuation usage
    punctuation = re.findall(r'[^\w\s]', text)
    punctuation_freq = Counter(punctuation)

    # Calculate part-of-speech tags
    pos_tags = pos_tag(words)
    pos_freq = Counter(tag for word, tag in pos_tags)

    word_entropy = calculate_entropy(word_freq, num_words)
    char_entropy = calculate_entropy(char_freq, num_chars)

    # Calculate capitalization patterns
    num_uppercase = sum(1 for c in text if c.isupper())
    num_lowercase = sum(1 for c in text if c.islower())
    capitalization_ratio = num_uppercase / num_lowercase if num_lowercase > 0 else 0

    # Compile features into a dictionary
    features = {
        'num_words': num_words,
        'num_chars': num_chars,
        'num_lines': num_lines,
        # 'word_freq': word_freq,
        # 'bigram_freq': bigram_freq,
        # 'char_freq': char_freq,
        # 'punctuation_freq': punctuation_freq,
        # 'pos_freq': pos_freq,
        'word_entropy': word_entropy,
        'char_entropy': char_entropy,
        'capitalization_ratio': capitalization_ratio
    }

    return features

In [3]:
df = pd.read_excel("data.xlsx")
df.head()

,Model,Input No,Output,QAL-Score,QAN-Score
0,GPT4o,1,uid=1000(user1) gid=1000(user1) groups=1000(us...,1.0,NaN
1,GPT4o,2,total 32\ndrwxr-xr-x 4 user1 user1 4096 Nov 2...,1.0,NaN
2,GPT4o,3,/home\ntotal 12\ndrwxr-xr-x 3 root root 409...,1.0,NaN
3,GPT4o,4,NaN,1.0,NaN
4,GPT4o,5,user1,0.0,NaN


In [4]:
df["Output"] = df["Output"].fillna("")

In [5]:
df["commandLen"] = df["Output"].apply(len)

In [6]:
response_metrics = []
for i in df.iterrows():
    response_metrics.append(calculate_features(i[1]["Output"]))


In [7]:
len(df), len(response_metrics)

(192, 192)

In [8]:
categories = response_metrics[0].keys()
categories
for i in categories:
    df[i] = 0

index = 0
for i in response_metrics:
    for c in categories:
        df[c].iloc[index] = i[c]
    index += 1

/var/folders/j1/9dq3kbp5329d222wgtb716ch0000gp/T/ipykernel_5101/2320074000.py:9: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[c].iloc[index] = i[c]
/var/folders/j1/9dq3kbp5329d222wgtb716ch0000gp/T/ipykernel_5101/2320074000.py:9: SettingW

In [9]:
df.head()

,Model,Input No,Output,QAL-Score,QAN-Score,commandLen,num_words,num_chars,num_lines,word_entropy,char_entropy,capitalization_ratio
0,GPT4o,1,uid=1000(user1) gid=1000(user1) groups=1000(us...,1.0,NaN,50,12,50,1,2.396241,3.670490,0.000000
1,GPT4o,2,total 32\ndrwxr-xr-x 4 user1 user1 4096 Nov 2...,1.0,NaN,406,89,406,9,4.325948,4.423147,0.121019
2,GPT4o,3,/home\ntotal 12\ndrwxr-xr-x 3 root root 409...,1.0,NaN,157,30,157,5,3.781076,4.266204,0.044776
3,GPT4o,4,,1.0,NaN,0,0,0,1,0.000000,0.000000,0.000000
4,GPT4o,5,user1,0.0,NaN,5,1,5,1,-0.000000,2.321928,0.000000


In [10]:
rf = joblib.load("../1-RFClassifier/RF.pkl")

In [11]:
keys = ["commandLen"]
for i in categories:
    keys.append(i)

predictions = rf.predict(df[keys])

In [12]:
df["QAN-Score"] = predictions

In [13]:
df.head()

,Model,Input No,Output,QAL-Score,QAN-Score,commandLen,num_words,num_chars,num_lines,word_entropy,char_entropy,capitalization_ratio
0,GPT4o,1,uid=1000(user1) gid=1000(user1) groups=1000(us...,1.0,0,50,12,50,1,2.396241,3.670490,0.000000
1,GPT4o,2,total 32\ndrwxr-xr-x 4 user1 user1 4096 Nov 2...,1.0,0,406,89,406,9,4.325948,4.423147,0.121019
2,GPT4o,3,/home\ntotal 12\ndrwxr-xr-x 3 root root 409...,1.0,1,157,30,157,5,3.781076,4.266204,0.044776
3,GPT4o,4,,1.0,0,0,0,0,1,0.000000,0.000000,0.000000
4,GPT4o,5,user1,0.0,1,5,1,5,1,-0.000000,2.321928,0.000000


In [14]:
for i in df["Model"].unique():
    mean_val = df[df["Model"] == i]["QAN-Score"].mean()
    print(f"{i}: {mean_val}")

GPT4o: 0.3125
GPT4omini: 0.375
Gemini1.5Flash: 0.0
SonarPro: 0.125
Sonar: 0.1875
DeepSeekR1: 0.0625
Phi4: 0.3125
Llama3-8: 0.4375
Llama3.1-8: 0.4375
Llama3.2-3: 0.1875
Llama3.2-1: 0.5
Llama3.3: 0.1875


In [15]:
for i in df["Model"].unique():
    mean_val = df[df["Model"] == i]["QAL-Score"].mean()
    print(f"{i}: {mean_val}")

GPT4o: 0.875
GPT4omini: 0.75
Gemini1.5Flash: 0.0
SonarPro: 0.6875
Sonar: 0.6875
DeepSeekR1: 0.4375
Phi4: 0.3125
Llama3-8: 0.25
Llama3.1-8: 0.15625
Llama3.2-3: 0.0625
Llama3.2-1: 0.0625
Llama3.3: 0.71875


In [16]:
df.to_excel("data_full.xlsx")